# Laboratório de Ciência de Dados
## Aula 10 — Detecção de anomalias II: Isolation Forest, LOF e One-Class SVM

Este notebook dá continuidade à Aula 9.

Na primeira parte, utilizamos IQR, escore robusto, distância de Mahalanobis e erro de reconstrução com PCA. Agora trabalharemos com algoritmos desenvolvidos especificamente para detectar observações incomuns em espaços multidimensionais.

Os três métodos estudados utilizam ideias diferentes:

- **Isolation Forest** procura observações que podem ser isoladas rapidamente;
- **Local Outlier Factor (LOF)** compara a densidade de uma observação com a densidade de seus vizinhos;
- **One-Class SVM** procura delimitar uma região que represente o comportamento considerado normal.

### Objetivos

Ao final da atividade, você deverá ser capaz de aplicar os três métodos, interpretar seus principais parâmetros, trabalhar com escores de anomalia, comparar detectores e investigar observações sinalizadas por algoritmos diferentes.

# 1. Preparação do ambiente

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score, recall_score, f1_score

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# 2. Base de transações

Utilizaremos uma nova versão da base da aula anterior. A coluna `anomalia_real` será usada somente na avaliação final. Nenhum dos algoritmos receberá esse rótulo durante o ajuste.

In [ ]:
n_normais = 1800
n_anomalias = 90

normais = pd.DataFrame({
    "valor": rng.lognormal(mean=4.9, sigma=0.55, size=n_normais),
    "transacoes_1h": np.clip(rng.poisson(1.2, n_normais), 0, 7),
    "distancia_habitual_km": np.clip(rng.exponential(9, n_normais), 0, 70),
    "idade_conta_dias": np.clip(rng.gamma(3.5, 450, n_normais), 30, 5000),
    "falhas_login_24h": np.clip(rng.poisson(0.25, n_normais), 0, 5),
    "uso_limite_pct": np.clip(rng.beta(2.0, 3.2, n_normais), 0, 1),
    "anomalia_real": 0
})

anomalias = pd.DataFrame({
    "valor": rng.lognormal(mean=6.8, sigma=0.75, size=n_anomalias),
    "transacoes_1h": rng.integers(4, 12, size=n_anomalias),
    "distancia_habitual_km": rng.uniform(50, 350, size=n_anomalias),
    "idade_conta_dias": rng.uniform(10, 700, size=n_anomalias),
    "falhas_login_24h": rng.integers(1, 8, size=n_anomalias),
    "uso_limite_pct": rng.uniform(0.65, 1.0, size=n_anomalias),
    "anomalia_real": 1
})

df = pd.concat([normais, anomalias], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

df["valor"] = df["valor"].round(2)
df["distancia_habitual_km"] = df["distancia_habitual_km"].round(2)
df["idade_conta_dias"] = df["idade_conta_dias"].round(0)
df["uso_limite_pct"] = df["uso_limite_pct"].round(3)

df.head()

In [ ]:
print("Observações:", len(df))
print("Anomalias reais:", int(df["anomalia_real"].sum()))
print("Percentual:", round(100 * df["anomalia_real"].mean(), 2), "%")

# 3. Padronização

LOF e One-Class SVM são sensíveis à escala dos atributos. Utilizaremos a mesma representação padronizada nos três métodos para facilitar a comparação.

In [ ]:
atributos = [
    "valor",
    "transacoes_1h",
    "distancia_habitual_km",
    "idade_conta_dias",
    "falhas_login_24h",
    "uso_limite_pct"
]

X = df[atributos]

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

pd.DataFrame(
    X_std,
    columns=atributos
).agg(["mean", "std"]).round(3)

# 4. Outlier detection e novelty detection

Há duas situações que aparecem com frequência.

Em **outlier detection**, procuramos observações incomuns dentro do próprio conjunto disponível. É o que faremos na maior parte desta aula.

Em **novelty detection**, o modelo aprende uma referência de comportamento normal e depois avalia novas observações que chegam no futuro. Nesse cenário, idealmente o conjunto usado como referência deve conter predominantemente exemplos normais.

Essa distinção é especialmente importante no LOF. Por padrão, `LocalOutlierFactor` foi desenhado para identificar outliers no conjunto analisado. Para avaliar novas observações, existe o modo `novelty=True`, que deve ser usado com cuidados específicos.

# 5. Isolation Forest

O Isolation Forest não procura diretamente um centro ou uma densidade. Ele cria árvores com divisões aleatórias. Observações incomuns tendem a ser isoladas após poucas divisões, enquanto pontos em regiões densas costumam exigir caminhos maiores.

O parâmetro `contamination` é utilizado na transformação do escore contínuo em rótulo de anomalia.

In [ ]:
iso = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rotulos_iso = iso.fit_predict(X_std)

df["anomalia_iso"] = (
    rotulos_iso == -1
).astype(int)

# Score maior = maior anormalidade.
df["score_iso"] = -iso.decision_function(X_std)

print("Sinalizadas:", int(df["anomalia_iso"].sum()))

In [ ]:
df[
    atributos + ["score_iso", "anomalia_real"]
].sort_values(
    "score_iso",
    ascending=False
).head(10)

### Questão 1

Por que uma observação isolada rapidamente tende a receber um escore de anomalia maior no Isolation Forest?

# 6. Local Outlier Factor — LOF

O LOF compara a densidade local de uma observação com a densidade encontrada ao redor de seus vizinhos.

Isso é importante porque uma observação pode estar distante do centro global e ainda assim ser normal se estiver inserida em uma região com outras observações semelhantes.

O parâmetro `n_neighbors` define o tamanho da vizinhança utilizada nessa comparação.

In [ ]:
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)

rotulos_lof = lof.fit_predict(X_std)

df["anomalia_lof"] = (
    rotulos_lof == -1
).astype(int)

df["score_lof"] = -lof.negative_outlier_factor_

print("Sinalizadas:", int(df["anomalia_lof"].sum()))

In [ ]:
df[
    atributos + ["score_lof", "anomalia_real"]
].sort_values(
    "score_lof",
    ascending=False
).head(10)

### Questão 2

Qual é a principal diferença entre perguntar se um ponto está longe do centro global e perguntar se sua densidade local é diferente da densidade de seus vizinhos?

# 7. One-Class SVM

A One-Class SVM procura construir uma fronteira que envolva a região correspondente ao comportamento considerado normal.

Com o kernel RBF, essa fronteira pode ser não linear. O parâmetro `nu` controla a flexibilidade da solução e se relaciona à fração de exceções admitidas pelo modelo.

In [ ]:
ocsvm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

rotulos_svm = ocsvm.fit_predict(X_std)

df["anomalia_svm"] = (
    rotulos_svm == -1
).astype(int)

df["score_svm"] = -ocsvm.decision_function(X_std)

print("Sinalizadas:", int(df["anomalia_svm"].sum()))

In [ ]:
df[
    atributos + ["score_svm", "anomalia_real"]
].sort_values(
    "score_svm",
    ascending=False
).head(10)

### Questão 3

Ao contrário do Isolation Forest, a One-Class SVM depende fortemente da geometria produzida pela escala dos atributos. Por que isso reforça a importância da padronização?

# 8. Comparação dos métodos

Como conhecemos os rótulos apenas por ser uma base simulada, podemos calcular métricas supervisionadas para estudar o comportamento dos detectores.

In [ ]:
def avaliar(nome, coluna):
    y_real = df["anomalia_real"]
    y_pred = df[coluna]

    return {
        "metodo": nome,
        "sinalizadas": int(y_pred.sum()),
        "precisao": precision_score(y_real, y_pred, zero_division=0),
        "recall": recall_score(y_real, y_pred, zero_division=0),
        "f1": f1_score(y_real, y_pred, zero_division=0)
    }


tabela = pd.DataFrame([
    avaliar("Isolation Forest", "anomalia_iso"),
    avaliar("LOF", "anomalia_lof"),
    avaliar("One-Class SVM", "anomalia_svm")
])

tabela.round(3)

### Questão 4

O mesmo método apresenta simultaneamente a maior precisão e o maior recall? Se não, qual compromisso aparece nos resultados?

Nesta base, é possível que o LOF apresente desempenho bem inferior ao Isolation Forest. Isso não indica necessariamente erro de implementação.

As anomalias simuladas foram geradas de forma semelhante entre si e acabam formando uma região relativamente coerente. Como o LOF compara cada observação principalmente com sua vizinhança, vários desses pontos podem parecer normais **dentro da própria região anômala**.

Esse resultado é útil para compreender a diferença entre métodos: o que parece uma anomalia global pode não ser uma anomalia local.

# 9. Concordância entre detectores

Os algoritmos representam conceitos diferentes de anormalidade. Por isso, não é esperado que sinalizem exatamente as mesmas observações.

In [ ]:
df["quantos_detectores"] = (
    df[
        [
            "anomalia_iso",
            "anomalia_lof",
            "anomalia_svm"
        ]
    ].sum(axis=1)
)

df["quantos_detectores"].value_counts().sort_index()

In [ ]:
df[
    atributos
    + [
        "anomalia_iso",
        "anomalia_lof",
        "anomalia_svm",
        "quantos_detectores",
        "score_iso",
        "anomalia_real"
    ]
].sort_values(
    ["quantos_detectores", "score_iso"],
    ascending=[False, False]
).head(15)

Uma observação sinalizada pelos três métodos merece atenção, mas essa concordância não constitui prova de fraude ou erro.

Da mesma forma, uma observação identificada por apenas um método não deve ser descartada sem análise. Ela pode refletir precisamente a característica que diferencia aquele algoritmo.

# 10. Visualização em duas dimensões com PCA

A base possui seis atributos. Utilizaremos PCA apenas para produzir uma projeção bidimensional e visualizar onde aparecem os alertas.

Os detectores continuarão trabalhando no espaço de seis dimensões.

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_std)

print(
    "Variância explicada em 2D:",
    round(
        100 * pca.explained_variance_ratio_.sum(),
        2
    ),
    "%"
)

In [ ]:
for nome, coluna in {
    "Isolation Forest": "anomalia_iso",
    "LOF": "anomalia_lof",
    "One-Class SVM": "anomalia_svm"
}.items():

    plt.figure(figsize=(7, 5))

    normais_plot = df[coluna] == 0
    anomalias_plot = df[coluna] == 1

    plt.scatter(
        X_2d[normais_plot, 0],
        X_2d[normais_plot, 1],
        alpha=0.35,
        label="normal"
    )

    plt.scatter(
        X_2d[anomalias_plot, 0],
        X_2d[anomalias_plot, 1],
        marker="x",
        s=55,
        label="anomalia"
    )

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(nome)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

### Questão 5

Alguns alertas parecem estar no interior da nuvem quando observados em duas dimensões?

Por que uma projeção PCA 2D pode esconder parte da razão pela qual o detector considerou uma observação anômala?

# 11. Efeito de `contamination` no Isolation Forest

`contamination` não muda apenas a apresentação do resultado; ele influencia o limiar usado para transformar os escores em rótulos.

Vamos comparar diferentes valores.

In [ ]:
resultados_contamination = []

for contamination in [
    0.01,
    0.03,
    0.05,
    0.08,
    0.10
]:
    modelo = IsolationForest(
        n_estimators=250,
        contamination=contamination,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    pred = (
        modelo.fit_predict(X_std) == -1
    ).astype(int)

    resultados_contamination.append({
        "contamination": contamination,
        "sinalizadas": int(pred.sum()),
        "precisao": precision_score(
            df["anomalia_real"],
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            df["anomalia_real"],
            pred,
            zero_division=0
        ),
        "f1": f1_score(
            df["anomalia_real"],
            pred,
            zero_division=0
        )
    })

pd.DataFrame(resultados_contamination).round(3)

### Questão 6

O que acontece com recall quando `contamination` aumenta? E com a quantidade de alertas?

# 12. Efeito de `n_neighbors` no LOF

Uma vizinhança pequena enfatiza padrões muito locais. Uma vizinhança maior compara cada ponto com uma região mais ampla.

In [ ]:
resultados_vizinhos = []

for k in [
    5,
    10,
    20,
    40,
    80
]:
    modelo = LocalOutlierFactor(
        n_neighbors=k,
        contamination=0.05
    )

    pred = (
        modelo.fit_predict(X_std) == -1
    ).astype(int)

    resultados_vizinhos.append({
        "n_neighbors": k,
        "precisao": precision_score(
            df["anomalia_real"],
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            df["anomalia_real"],
            pred,
            zero_division=0
        ),
        "f1": f1_score(
            df["anomalia_real"],
            pred,
            zero_division=0
        )
    })

pd.DataFrame(resultados_vizinhos).round(3)

### Questão 7

O desempenho do LOF é invariável ao número de vizinhos? O que muda conceitualmente quando aumentamos a vizinhança?

# 13. Efeito de `nu` na One-Class SVM

Vamos variar `nu` mantendo o kernel RBF.

In [ ]:
resultados_nu = []

for nu in [
    0.01,
    0.03,
    0.05,
    0.08,
    0.10
]:
    modelo = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=nu
    )

    pred = (
        modelo.fit_predict(X_std) == -1
    ).astype(int)

    resultados_nu.append({
        "nu": nu,
        "sinalizadas": int(pred.sum()),
        "precisao": precision_score(
            df["anomalia_real"],
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            df["anomalia_real"],
            pred,
            zero_division=0
        ),
        "f1": f1_score(
            df["anomalia_real"],
            pred,
            zero_division=0
        )
    })

pd.DataFrame(resultados_nu).round(3)

# 14. Um exemplo de anomalia local

Para entender melhor o LOF, vamos construir um conjunto bidimensional com duas regiões normais de densidades diferentes e alguns pontos isolados.

In [ ]:
grupo_denso = rng.normal(
    loc=[0, 0],
    scale=[0.45, 0.45],
    size=(350, 2)
)

grupo_esparso = rng.normal(
    loc=[5, 5],
    scale=[1.2, 1.2],
    size=(120, 2)
)

pontos_isolados = np.array([
    [1.8, 0.2],
    [5.0, 8.5],
    [8.5, 5.0],
    [-2.0, 1.5]
])

X_local = np.vstack([
    grupo_denso,
    grupo_esparso,
    pontos_isolados
])

plt.figure(figsize=(7, 5))
plt.scatter(
    X_local[:, 0],
    X_local[:, 1],
    alpha=0.6
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Regiões normais com densidades diferentes")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
lof_local = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.02
)

labels_local = (
    lof_local.fit_predict(X_local) == -1
)

plt.figure(figsize=(7, 5))

plt.scatter(
    X_local[~labels_local, 0],
    X_local[~labels_local, 1],
    alpha=0.45,
    label="normal"
)

plt.scatter(
    X_local[labels_local, 0],
    X_local[labels_local, 1],
    marker="x",
    s=90,
    label="LOF"
)

plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Detecção baseada em densidade local")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

O exemplo mostra por que "estar longe do centro global" e "estar em uma região localmente incomum" são conceitos diferentes.

# 15. Inspeção de casos

Em aplicações reais, uma lista de alertas geralmente precisa ser investigada.

Vamos selecionar as observações com maior score no Isolation Forest e comparar seus atributos com as medianas da base.

In [ ]:
medianas = df[atributos].median()

top_iso = df.sort_values(
    "score_iso",
    ascending=False
).head(8).copy()

display(
    top_iso[
        atributos
        + [
            "score_iso",
            "anomalia_real"
        ]
    ]
)

print("\nMedianas da base:")
display(
    medianas.to_frame("mediana").T
)

### Questão 8

Escolha duas observações da tabela e descreva quais atributos parecem contribuir para que elas sejam diferentes do comportamento típico.

# 16. Exercícios

## Exercício 1 — Isolation Forest

Teste `contamination=0.02`, `0.04`, `0.06` e `0.08`. Construa uma tabela com quantidade sinalizada, precisão, recall e F1.

In [ ]:
# Complete aqui.

## Exercício 2 — LOF

Compare `n_neighbors=10`, `30` e `60`. Identifique observações que mudam de classificação entre as configurações.

In [ ]:
# Complete aqui.

## Exercício 3 — One-Class SVM

Compare `gamma="scale"`, `0.01`, `0.1` e `1.0`, mantendo `nu=0.05`. Analise a quantidade de alertas e o F1.

In [ ]:
# Complete aqui.

## Exercício 4 — consenso

Crie três categorias:

- detectada por nenhum algoritmo;
- detectada por exatamente um ou dois algoritmos;
- detectada pelos três.

Compare a taxa de `anomalia_real` em cada categoria.

In [ ]:
# Complete aqui.

# 17. Desafio final

Você recebeu autorização para encaminhar apenas 30 transações para investigação manual.

Construa uma estratégia de priorização utilizando os escores ou a concordância entre os três detectores.

Depois informe:

1. quais 30 transações seriam selecionadas;
2. quantas delas são anomalias reais na base simulada;
3. qual critério foi utilizado;
4. por que esse critério seria defensável;
5. quais limitações ele possui.

Não existe uma única solução correta. A justificativa deve ser coerente com os resultados observados.

In [ ]:
# Complete aqui.

# 18. Questões de revisão

**1. Qual é a ideia central do Isolation Forest?**

**2. O que significa densidade local no LOF?**

**3. Para que serve `n_neighbors`?**

**4. O que a One-Class SVM procura delimitar?**

**5. Qual é o papel de `contamination`?**

**6. Por que diferentes detectores podem discordar?**

**7. Uma observação sinalizada pelos três algoritmos é necessariamente fraude?**

**8. Por que uma projeção PCA em duas dimensões não explica completamente uma decisão tomada em seis dimensões?**

**9. Qual é a diferença entre anomalia global e local?**

**10. Como você avaliaria um detector em uma aplicação real sem rótulos confiáveis?**

# 19. Conclusão

Nesta segunda aula, a detecção de anomalias foi analisada com métodos que representam conceitos diferentes de comportamento incomum.

O Isolation Forest identifica observações fáceis de isolar. O LOF compara densidades locais e pode reconhecer desvios relativos à vizinhança. A One-Class SVM constrói uma fronteira para a região considerada normal.

A comparação mostrou que não existe um detector universalmente superior. A escolha depende da geometria dos dados, do tipo de anomalia de interesse, dos parâmetros utilizados e do custo associado aos alertas.